In [1]:
import tensorflow as tf
import sklearn
import pandas as pd
import numpy as np
from tensorflow.keras import layers, models
import time

ImportError: DLL load failed while importing _pywrap_tf2: The specified module could not be found.

In [ ]:
SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

NameError: name 'np' is not defined

In [ ]:
from datasets import load_dataset

# Load Huggingface Dataset
dataset = load_dataset("SetFit/bbc-news")

In [ ]:
# Load Train Dataset
train = dataset["train"].to_pandas()

# Load Test Dataset
test = dataset["test"].to_pandas()

In [ ]:
print("\nMissing values:")
print(train[["text", "label"]].isnull().sum())
print(test[["text", "label"]].isnull().sum())

### Mapping Label
0 : tech

1 : business

2 : sport

3 : entertainment

4 : politics

In [ ]:
print("\nTraining label mapping:")
print(
    train[["label", "label_text"]]
    .drop_duplicates()
    .sort_values("label")
)

In [ ]:
# Validation Split
from sklearn.model_selection import train_test_split

X = train["text"].to_numpy()
y = train["label"].to_numpy()

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

X_test = test["text"].to_numpy()
y_test = test["label"].to_numpy()

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

In [ ]:
MAX_WORDS = 10000
MAX_LEN = 300

tokenizer = tf.keras.preprocessing.text.Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = tf.keras.preprocessing.sequence.pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_val_pad = tf.keras.preprocessing.sequence.pad_sequences(
    X_val_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_pad = tf.keras.preprocessing.sequence.pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

print("Training input shape:", X_train_pad.shape)
print("Validation input shape:", X_val_pad.shape)
print("Test input shape:", X_test_pad.shape)
print("Vocabulary size:", min(len(tokenizer.word_index) + 1, MAX_WORDS))

In [ ]:
NUM_CLASSES = train["label"].nunique()

print("Number of classes:", NUM_CLASSES)
print("Training labels:", np.unique(y_train))
print("Test labels:", np.unique(y_test))

# RNN

In [ ]:
rnn_model = models.Sequential([
    layers.Input(shape=(MAX_LEN,)),

    layers.Embedding(
        input_dim=MAX_WORDS,
        output_dim=64,
        mask_zero=True
    ),

    layers.SpatialDropout1D(0.2),
    layers.SimpleRNN(32, dropout=0.2, recurrent_dropout=0.2),

    layers.Dense(32, activation="relu"),
    layers.Dropout(0.5),

    layers.Dense(NUM_CLASSES, activation='softmax')
])

rnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=['accuracy']
)

rnn_model.summary()

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

start_time = time.perf_counter()

history = rnn_model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=15,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

training_time = time.perf_counter() - start_time

print(f"Training Time : {training_time:.2f} seconds")

In [ ]:
# Save Model
rnn_model.save("rnn_model.keras")

# Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
test_loss, test_accuracy = rnn_model.evaluate(
    X_test_pad,
    y_test,
    verbose=0
)

start_time_eval = time.perf_counter()

y_pred_prob = rnn_model.predict(X_test_pad, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)

inference_time = time.perf_counter() - start_time_eval

# Evaluation metrics
majority_baseline = np.bincount(y_train).max() / len(y_train)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Majority-class baseline: {majority_baseline:.4f}")
print(f"Macro Precision: {precision_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"Macro Recall: {recall_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"Macro F1: {f1_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"Inference Time: {inference_time:.4f} seconds")

In [ ]:
label_mapping = (
    train[["label", "label_text"]]
    .drop_duplicates()
    .sort_values("label")
)
label_names = label_mapping["label_text"].to_list()

print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=label_names,
    zero_division=0
))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

label_mapping = (
    train[["label", "label_text"]]
    .drop_duplicates()
    .sort_values("label")
)
label_names = label_mapping["label_text"].to_list()

cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2, 3, 4])

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_names,
    yticklabels=label_names
)

plt.title("RNN Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()